# k2 — Dịch 1.200 câu hỏi vi → en → zh, so sánh hai model (GPU T4 ×2)

**Cài đặt:** Accelerator = **GPU T4 ×2**, Internet = On. **Add Input:** `mir2-code`, `mir2-data`
(+ output phiên trước của notebook này nếu cần chạy tiếp).

Mỗi model chạy 2 tiến trình song song (mỗi GPU một bản 4-bit, mỗi bản một nửa số câu). Qwen3 không có bản 7B →
dùng **Qwen3-8B**; muốn so cùng cỡ thêm `Qwen/Qwen3-14B` vào `MODELS`. Ước tính (chưa đo): 14B ~1,5–2,5 giờ, 8B ~1 giờ.
Ô (2) chạy thử 20 câu trước (đặt `LIMIT = 20`) để đọc mẫu, rồi `LIMIT = 0` cho đủ 1.200 câu.

**Đầu ra:** `qtrans/<model>/records-*.jsonl`, `qtrans/compare_*/{auto_metrics.md, sheet.csv, key.json}` → dataset
**`mir2-qtrans`**. Phiếu `sheet.csv` tải về cho 2–3 người chấm mù (không mở `key.json`), rồi chấm điểm trên máy:
`python -m mirret.translate score --sheet <phiếu đã chấm>` (xem docs/EVAL_PLAN.md mục 0).

In [ ]:
# (1) Chuẩn bị. Chỉ sửa các biến viết HOA ngay dưới đây.
NB_VERSION = 'k2-v1 (15/10/2026)'
MODELS = ['qwen25_14b', 'qwen3_8b']   # tên trong translate.models hoặc id Hugging Face
LIMIT = 0
import glob, os, shutil, sys, zipfile
KAGGLE = os.environ.get('KAGGLE_ROOT', '/kaggle')            # chạy thử ngoài Kaggle: đặt biến môi trường KAGGLE_ROOT
CODE = f'{KAGGLE}/working/code'
shutil.rmtree(CODE, ignore_errors=True)
hits = sorted(glob.glob(f'{KAGGLE}/input/**/mirret/__init__.py', recursive=True), key=len)
if hits:
    shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE)
else:
    zips = sorted(glob.glob(f'{KAGGLE}/input/**/mir2_code.zip', recursive=True))
    assert zips, 'Chưa thêm dataset mir2-code (Add Input → Your Datasets)'
    zipfile.ZipFile(zips[0]).extractall(CODE)
os.chdir(CODE); sys.path.insert(0, CODE)
from mirret.kaggle_nb import setup, sh, par, locate, locate_all, report
K = setup(NB_VERSION, {}, inputs=())

In [ ]:
# (2) Chép bản ghi của phiên trước (nếu có) để chạy tiếp; cài bitsandbytes (4-bit)
for d in locate_all('qtrans'):
    for f in glob.glob(f'{d}/qtrans/*/records*.jsonl'):
        dst = f'{K.work}/qtrans/' + os.path.relpath(f, f'{d}/qtrans')
        if not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(f, dst); print('chép', f)
sh('python -m pip install -q -U bitsandbytes accelerate')

In [ ]:
# (3) Sinh bản dịch cho từng model, 2 GPU song song
for m in MODELS:
    lim = f' --limit {LIMIT}' if LIMIT else ''
    par([f'CUDA_VISIBLE_DEVICES={g} python -m mirret.translate generate --config {K.cfg} --model {m} --shard {g}/2{lim}'
         for g in range(2)], f'translate_{m}')

In [ ]:
# (4) So sánh: chỉ số tự động (hình thức) + phiếu chấm mù cho người
if len(MODELS) >= 2:
    sh(f'python -m mirret.translate compare --config {K.cfg} --a {MODELS[0]} --b {MODELS[1]} --n 100')
report('k2')